In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tarfile
import zipfile
import bz2
import json

import pandas as pd
from IPython.display import display

In [2]:
BASE_URL = "https://data.everef.net"

urls = {
    "market_history": "market-history/2026/market-history-2026-01-01.csv.bz2",
    "market_orders": "market-orders/history/2026/2026-01-01/market-orders-2026-01-01_05-15-07.v3.csv.bz2",
    "market_prices": "markets-prices/history/2026/2026-01-01/markets-prices-2026-01-01_00-27-01.json.bz2",
    "fuzzwork": "fuzzwork/ordersets/2026/2026-01-01/fuzzwork-orderset-161676-2026-01-01_12-06-49.csv.gz",
    "killmail": "killmails/2026/killmails-2026-01-01.tar.bz2",
    "public_contracts": "public-contracts/history/2026/2026-01-01/public-contracts-2026-01-01_01-00-07.v2.tar.bz2",
    "mer": "ccp/mer/2026/EVEOnline_MER_202601.zip",
    "refdata": "reference-data/reference-data-latest.tar.xz",
    "wars": "wars/history/2026/wars-2026-01-01_00-13-06.tar.bz2"
}

REPOSITORY_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'experiments' / 'everef' / 'endpoint-exploration.ipynb').is_file()
)
local_dir = REPOSITORY_ROOT / 'experiments' / '.local'
local_dir.mkdir(exist_ok=True)

In [3]:
for name, rel_url in urls.items():
    url = f"{BASE_URL}/{rel_url}"
    out = local_dir / Path(rel_url).name

    print(name, "->", out)

    if not out.exists():
        urlretrieve(url, out)
    else:
        print("already exists")

In [4]:
market_history_path = local_dir / Path(urls["market_history"]).name
market_history = pd.read_csv(market_history_path)

market_history.head(20)

In [5]:
market_orders_path = local_dir / Path(urls["market_orders"]).name
market_orders = pd.read_csv(market_orders_path)

market_orders.head(20)

In [6]:
market_orders["order_id"].is_unique

## Joining market datasets to reference data

Market-facing Everef datasets such as `market_history`, `market_orders`, `market_prices`, and Fuzzwork ordersets all expose `type_id`. That is the stable bridge into Everef reference data.

Recommended joins:

- `market_* .type_id -> refdata/types.json[type_id]`
- `types.group_id -> groups.json[group_id]`
- `groups.category_id -> categories.json[category_id]`
- `types.market_group_id -> market_groups.json[market_group_id]` when present

Important distinction: `group_id` and `market_group_id` are different foreign keys in different namespaces. `group_id` is inventory classification, while `market_group_id` is market-browser classification. They should not be joined to each other.

Primary sources:

- Everef reference-data docs: https://docs.everef.net/datasets/reference-data
- Official ESI type examples: https://esi.evetech.net/latest/universe/types/34/?datasource=tranquility and https://esi.evetech.net/latest/universe/types/587/?datasource=tranquility
- EVE Ref type pages: https://everef.net/types/34 and https://everef.net/types/587


Fuzzworks possible columns:

Source: https://github.com/fuzzysteve/FuzzMarket/blob/master/scripts/aggloader-esi.py

1. `order_id` - is this definitely unique? hmmmm
2. `type_id`
3. `issued` timestamp
4. `is_buy_order` boolean
5. `volume_remain`
6. `volume_total`
7. `min_volume` - minimum quantity to fill the order (is it ever not 1?)
8. `price` - ISK
9. `location_id` - ESI location(?)
10. `range` - in jumps(?)
11. `duration` in days
12. `region_id` - added by Fuzzwork
13. `order_set_id` - added by Fuzzwork

In [7]:
fuzzwork_path = local_dir / Path(urls["fuzzwork"]).name

fuzzwork_order_cols = [
    "order_id",
    "type_id",
    "issued",
    "is_buy_order",
    "volume_remain",
    "volume_total",
    "min_volume",
    "price",
    "location_id",
    "range",
    "duration",
    "region_id",
    "order_set_id",
]

# Fuzzwork orderset is tab-separated and has no first header line.
fuzzwork = pd.read_csv(fuzzwork_path, sep="\t", header=None, names=fuzzwork_order_cols)

fuzzwork.head(20)

In [8]:
market_prices_path = local_dir / Path(urls["market_prices"]).name
market_prices_dir = local_dir / "market_prices"
market_prices_dir.mkdir(exist_ok=True)

market_prices_json_path = market_prices_dir / market_prices_path.with_suffix("").name

with bz2.open(market_prices_path, "rb") as src:
    market_prices_json_path.write_bytes(src.read())

market_prices_json_path

In [9]:
with open(market_prices_json_path, "r") as f:
    market_prices_json = json.load(f)

type(market_prices_json), len(market_prices_json)

In [10]:
market_prices = pd.DataFrame(market_prices_json)
market_prices.head()

In [11]:
wars_path = local_dir / Path(urls["wars"]).name

if not wars_path.exists():
    urlretrieve(f"{BASE_URL}/{urls['wars']}", wars_path)

wars_dir = local_dir / "wars"
wars_dir.mkdir(exist_ok=True)

with tarfile.open(wars_path) as tf:
    tf.extractall(wars_dir, filter="data")

list(wars_dir.rglob("*"))[:20]

In [12]:
killmail_path = local_dir / Path(urls["killmail"]).name
killmail_dir = local_dir / "killmail"
killmail_dir.mkdir(exist_ok=True)

with tarfile.open(killmail_path) as tf:
    tf.extractall(killmail_dir, filter="data")

list(killmail_dir.iterdir())[:10]

In [13]:
public_contracts_path = local_dir / Path(urls["public_contracts"]).name
public_contracts_dir = local_dir / "public_contracts"
public_contracts_dir.mkdir(exist_ok=True)

with tarfile.open(public_contracts_path) as tf:
    tf.extractall(public_contracts_dir, filter="data")

list(public_contracts_dir.iterdir())[:10]

In [14]:
refdata_path = local_dir / Path(urls["refdata"]).name
refdata_dir = local_dir / "refdata"
refdata_dir.mkdir(exist_ok=True)

with tarfile.open(refdata_path) as tf:
    tf.extractall(refdata_dir, filter="data")

list(refdata_dir.iterdir())[:20]

## Everef reference-data relationship map

Everef publishes separate reference-data endpoints and files for inventory groups and market groups. Those are not interchangeable.

Core relationship chain from market datasets:

- `market_history.type_id -> types.type_id`
- `market_orders.type_id -> types.type_id`
- `market_prices.type_id -> types.type_id`
- `fuzzwork.type_id -> types.type_id`

From `types` there are two parallel classification chains:

Inventory chain:

- `types.group_id -> groups.group_id`
- `groups.category_id -> categories.category_id`

Market tree chain:

- `types.market_group_id -> market_groups.market_group_id`
- `market_groups.parent_group_id -> market_groups.market_group_id` recursively until root

Practical interpretation:

- inventory chain answers: what kind of item is this?
- market tree chain answers: where does this appear in the market browser?

Cited sources:

- Everef reference-data docs list `/groups` from `groups.yaml` and `/market_groups` from `marketGroups.yaml`: https://docs.everef.net/datasets/reference-data
- ESI type 34 exposes both `group_id` and `market_group_id`: https://esi.evetech.net/latest/universe/types/34/?datasource=tranquility
- ESI type 587 exposes both `group_id` and `market_group_id`: https://esi.evetech.net/latest/universe/types/587/?datasource=tranquility
- EVE Ref type pages label separate inventory and market group hierarchies: https://everef.net/types/34 and https://everef.net/types/587

Next cells prove this locally without loading full `types.json` into memory. They scan only targeted records from keyed JSON files.


In [15]:
def _update_json_brace_state(text, brace_depth=0, in_string=False, escape=False):
    for char in text:
        if escape:
            escape = False
            continue

        if char == '\\' and in_string:
            escape = True
            continue

        if char == '\"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if char == '{':
            brace_depth += 1
        elif char == '}':
            brace_depth -= 1

    return brace_depth, in_string, escape


def read_keyed_json_record(path, record_key):
    marker = f'\"{record_key}\"'

    with open(path, 'r', encoding='utf-8') as f:
        collecting = False
        brace_depth = 0
        in_string = False
        escape = False
        buffer = []

        for line in f:
            if not collecting:
                if line.startswith(marker):
                    first_brace = line.index('{')
                    segment = line[first_brace:]
                    buffer.append(segment)
                    brace_depth, in_string, escape = _update_json_brace_state(segment)
                    collecting = True
                    if brace_depth == 0:
                        break
                continue

            buffer.append(line)
            brace_depth, in_string, escape = _update_json_brace_state(line, brace_depth, in_string, escape)
            if brace_depth == 0:
                break

    if not buffer:
        raise KeyError(f'{record_key} not found in {path}')

    record_text = ''.join(buffer).rstrip()
    if record_text.endswith(','):
        record_text = record_text[:-1]

    return json.loads(record_text)


types_path = refdata_dir / 'types.json'
groups_path = refdata_dir / 'groups.json'
categories_path = refdata_dir / 'categories.json'
market_groups_path = refdata_dir / 'market_groups.json'


In [16]:
proof_examples = []

for type_id in [2, 18, 41]:
    type_record = read_keyed_json_record(types_path, type_id)
    group_record = read_keyed_json_record(groups_path, type_record['group_id'])

    type_market_group_id = type_record.get('market_group_id')
    type_market_group_name = None
    if type_market_group_id is not None:
        type_market_group_name = read_keyed_json_record(market_groups_path, type_market_group_id)['name']['en']

    same_numeric_market_group_name = None
    try:
        same_numeric_market_group_name = read_keyed_json_record(market_groups_path, type_record['group_id'])['name']['en']
    except KeyError:
        pass

    proof_examples.append({
        'type_id': type_id,
        'type_name': type_record['name']['en'],
        'group_id': type_record['group_id'],
        'group_name': group_record['name']['en'],
        'type_market_group_id': type_market_group_id,
        'type_market_group_name': type_market_group_name,
        'same_numeric_market_group_name': same_numeric_market_group_name,
    })

pd.DataFrame(proof_examples)


In [17]:
def inventory_chain(type_id):
    type_record = read_keyed_json_record(types_path, type_id)
    group_record = read_keyed_json_record(groups_path, type_record['group_id'])
    category_record = read_keyed_json_record(categories_path, group_record['category_id'])

    return pd.DataFrame([
        {'level': 'type', 'id': type_record['type_id'], 'name': type_record['name']['en']},
        {'level': 'group', 'id': group_record['group_id'], 'name': group_record['name']['en']},
        {'level': 'category', 'id': category_record['category_id'], 'name': category_record['name']['en']},
    ])


def market_chain(type_id):
    type_record = read_keyed_json_record(types_path, type_id)
    market_group_id = type_record.get('market_group_id')

    if market_group_id is None:
        return pd.DataFrame([
            {'level': 'type', 'id': type_record['type_id'], 'name': type_record['name']['en']},
            {'level': 'market_group', 'id': None, 'name': None},
        ])

    chain = [{'level': 'type', 'id': type_record['type_id'], 'name': type_record['name']['en']}]
    current_id = market_group_id

    while current_id is not None:
        record = read_keyed_json_record(market_groups_path, current_id)
        chain.append({'level': 'market_group', 'id': record['market_group_id'], 'name': record['name']['en']})
        current_id = record.get('parent_group_id')

    return pd.DataFrame(chain)


for type_id in [34, 587]:
    print(f'Type {type_id} inventory chain')
    display(inventory_chain(type_id))
    print(f'Type {type_id} market chain')
    display(market_chain(type_id))


In [18]:
mer_path = local_dir / Path(urls["mer"]).name
mer_dir = local_dir / "mer"
mer_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(mer_path) as zf:
    zf.extractall(mer_dir)

list(mer_dir.iterdir())[:20]

In [19]:
list(killmail_dir.rglob("*"))[:20]

In [20]:
list(public_contracts_dir.rglob("*"))[:20]

In [21]:
list(refdata_dir.rglob("*"))[:20]

In [22]:
list(mer_dir.rglob("*"))[:20]